# Acoustic–text fusion on full Casablanca validation

Evaluate 4K and 8K acoustic-only heads, text-only, and fixed fusion on the **same 6,819 Casablanca validation rows** as the previous scaling run and the same 200 ADI-20 dev clips. The published 8K acoustic reference was **59.31% accuracy and 63.81% macro-F1**. One unreadable Casablanca row remains in the denominator.

1. Add your existing **ADI20_Casablanca_Fusion_Ready_Inputs.zip** as an input. Attach any previous `reusable_adi_cache` output containing Casablanca and ADI-20 dev features. Use **GPU T4 ×2** with **Internet** for missing features or transcripts.
2. The text head defaults to the same 500 training transcripts and fixed weight **1.3** as the prior 800-clip run. It reuses the 800 Casablanca transcript records, transcribes the remaining **6,018 readable clips** with pinned Jais, and keeps the original transcript-failure fallback. This does not train the acoustic heads.
3. Run all cells. **adi20_casablanca_full_fusion_results.zip** is written once both splits finish. The result includes paired metrics, confusion matrices, and the number of acoustic predictions differing from the archived baseline.
4. The notebook checks a **6-hour cap** for Casablanca transcription and feature extraction. If it pauses, attach `casablanca_full_jais_progress.jsonl` and any new `reusable_adi_cache` files from Output, together with your original inputs. The progress file also lets future runs skip the completed ASR. CPU works when all required transcripts and acoustic features are cached.

Optional `TEXT_TRAIN_SIZE = 2000` uses the fixed 2K text split. Attach `train2000_transcripts.jsonl` to avoid the separate 1,500-clip ADI-20 training transcription step.


## 1. Set up the pinned repository and load the inputs


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import BadZipFile, ZIP_DEFLATED, ZipFile
import json
import re

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
COMMIT = "b64437a37e7eda2f1721a56860278640bf085bdd"
BUNDLE_NAME = "ADI20_Casablanca_Fusion_Ready_Inputs.zip"
HEAD_NAME = "adi20_4k_vs_8k_results.zip"
TEXT_ZIP = "ADI20_Casablanca_Fusion_Text_Inputs.zip"
TEXT_FILES = {
    "jais_train500_transcripts.jsonl", "jais_dev200_transcripts.jsonl",
    "casablanca_800_jais_transcripts.jsonl", "haidar_training_text_availability.jsonl",
}
SEEDS = [490, 491, 492]
HEAD_FILES = {"protocol.json"}
for arm in ("train_4000", "train_8000"):
    HEAD_FILES.add(f"{arm}/zscore_stats.npz")
    for seed in SEEDS:
        HEAD_FILES.add(f"{arm}/head_{seed}.pt")
        for split in ("ADI20_dev", "Casablanca"):
            HEAD_FILES.add(f"{arm}/{seed}/{split}_predictions.csv")


def load_head_protocol(data):
    protocol = json.loads(data)
    if (protocol["repo_commit"] != COMMIT or protocol["seeds"] != SEEDS
            or protocol["manifest_sha256"]["train_scaling_8000"]
            != "c3e90a5441c1d70596c635c0a5e76147c88101b1498c22e76f167090fcea842d"):
        raise ValueError("Checkpoint protocol differs from the frozen 4K/8K run.")
    return protocol


def checked_head_zip(path):
    try:
        with ZipFile(path) as archive:
            names = archive.namelist()
            if HEAD_FILES <= set(names) and len(names) == len(set(names)):
                return load_head_protocol(archive.read("protocol.json"))
    except (BadZipFile, KeyError, OSError, TypeError, ValueError):
        pass
    return None


def unzip_fusion_bundle(zip_files, target):
    # Accept both the flat bundle and the original version with the two inner ZIPs.
    flat_heads = {f"acoustic_heads/{name}" for name in HEAD_FILES}
    flat_text = {f"text_inputs/{name}" for name in TEXT_FILES}
    cache_pattern = re.compile(
        r"features/arabest-rq-300m-6k/dff453e2b5f7/e4b3c287854e40f9/"
        r"(?:config\.json|index\.jsonl|part-[a-f0-9]{16}\.npz)"
    )
    for path in zip_files:
        try:
            with ZipFile(path) as archive:
                members = archive.namelist()
                names = set(members)
                if len(names) != len(members):
                    continue
                if flat_heads | flat_text <= names:
                    selected = flat_heads | flat_text | {"text_inputs/text_input_provenance.json"}
                elif {HEAD_NAME, TEXT_ZIP} <= names:
                    selected = {HEAD_NAME, TEXT_ZIP}
                else:
                    continue
                selected |= {name for name in names if cache_pattern.fullmatch(name)}
                for name in sorted(selected & names):
                    archive.extract(name, target)
                print(f"Fusion input bundle: {path.name}")
                return
        except BadZipFile:
            continue


def resolve_fusion_inputs(input_root, work_root, target):
    files = sorted(path for path in input_root.rglob("*") if path.is_file())
    zip_files = sorted(
        (path for path in files if path.suffix.lower() == ".zip"),
        key=lambda path: (path.name.casefold() != BUNDLE_NAME.casefold(), path),
    )
    unzip_fusion_bundle(zip_files, target)
    input_roots = [target, input_root]

    # Prefer the supplied bundle. A separate copy of the same result is fine.
    staged_heads = target / "acoustic_heads"
    extracted = [staged_heads, *(path.parent for path in files if path.name == "protocol.json")]
    direct = [target / HEAD_NAME, work_root / HEAD_NAME, *zip_files]
    if staged_heads.is_dir():
        direct = [target / HEAD_NAME]
    for path in direct:
        if path.is_file() and (protocol := checked_head_zip(path)) is not None:
            print(f"Acoustic heads: {path.name} (4K and 8K, 3 seeds each)")
            return input_roots, path, protocol

    for folder in extracted:
        if not all((folder / name).is_file() for name in HEAD_FILES):
            continue
        try:
            protocol = load_head_protocol((folder / "protocol.json").read_bytes())
        except (KeyError, OSError, TypeError, ValueError):
            continue
        archive_path = target / HEAD_NAME
        with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
            for name in sorted(HEAD_FILES):
                archive.write(folder / name, arcname=name)
        if (folder.parent / "text_inputs").is_dir() and folder.parent != target:
            input_roots.insert(1, folder.parent)
        print(f"Acoustic heads: {folder.name}/ (4K and 8K, 3 seeds each)")
        return input_roots, archive_path, protocol

    mounted = zip_files or files
    preview = ", ".join(str(path.relative_to(input_root)) for path in mounted[:5]) or "no input files mounted"
    raise FileNotFoundError(
        f"No matching 4K/8K heads found. Attach {BUNDLE_NAME} using Kaggle's Add Input, "
        "then rerun this cell. You can also upload the extracted bundle as one private Dataset. "
        f"Mounted under {input_root}: {preview}."
    )


fusion_input_directory = TemporaryDirectory(prefix="adi20-casablanca-inputs-")
INPUT_ROOTS, head_archive, protocol = resolve_fusion_inputs(INPUT, WORK, Path(fusion_input_directory.name))


In [ ]:
import importlib.util
import os
import subprocess
import sys

REPO = Path("/tmp/adi-mixture-full-fusion")
REPO_URL = "https://github.com/alizbeeb5/Arabic-Dialect-Identification.git"

FUSION_WEIGHT = 1.3
FULL_EVAL_HOURS = 6.0
TEXT_TRAIN_SIZE = 500        # Set 2000 to use the larger text training split.
EXTRACT_MISSING = True       # Encode only evaluation clips absent from attached caches.
EXTRACT_BATCH = 4
EXTRACT_CHUNK = 48
PROBE_CLIPS = 200
SPARSE_HF_READS = True

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
WORK.mkdir(parents=True, exist_ok=True)

def git(*args):
    subprocess.run(["git", "-C", str(REPO), *args], check=True)

if not (REPO / ".git").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    git("init", "--quiet")
    git("remote", "add", "origin", REPO_URL)

pinned_commit = subprocess.run(
    ["git", "-C", str(REPO), "cat-file", "-e", f"{COMMIT}^{{commit}}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
if pinned_commit.returncode:
    git("fetch", "--quiet", "--depth=1", "origin", COMMIT)
git("checkout", "--quiet", "--detach", COMMIT)
sys.path.insert(0, str(REPO))

# Keep Kaggle's CUDA build of PyTorch and torchaudio.
packages = {"soundfile": "soundfile", "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub"}
missing_packages = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)


In [ ]:
import hashlib
import io
import shutil
import time
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from scipy.special import softmax
from sklearn.metrics import confusion_matrix, f1_score

from src.models import acoustic_encoder as ae
from src.experiments import acoustic_classifier as clf, layer_mixture as lm

torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32 = False

M = REPO / "data/manifests"
RUN = WORK / "full_fusion_runs" / datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
RUN.mkdir(parents=True, exist_ok=True)
LABELS = json.loads((REPO / "configs/labels.json").read_text())["label_order"]
ADI_REV = "c8da454c9f474bb075d3fe608940ee29e8ccfa43"
CAS_REV = "8951b1b88e28c1107142ced57967b8d16350951d"
CASABLANCA_LABELS = {"ALG", "EGY", "JOR", "MAU", "MOR", "PAL", "UAE", "YEM"}

def read_manifest(name):
    raw = (M / f"{name}.jsonl").read_bytes()
    expected = ("41514213e14bfbd82d4fbf8cfe6eae04a2faf4f878509a15758f431dc41f7929"
                if name == "train_scaling_2000" else protocol["manifest_sha256"][name])
    assert hashlib.sha256(raw).hexdigest() == expected, f"Manifest mismatch: {name}"
    return [json.loads(line) for line in raw.splitlines() if line.strip()]


## 2. Complete the Casablanca transcripts and fit the text classifier


In [ ]:
def find_input(name, required=False):
    local = WORK / name
    if local.is_file():
        return local
    paths = sorted({p for root in INPUT_ROOTS for p in root.rglob(name) if p.is_file()})
    if len(paths) > 1:
        raise ValueError(f"Found multiple copies of {name}; attach one version.")
    if paths:
        return paths[0]
    if required:
        raise FileNotFoundError(f"Attach {name} as a private Kaggle input.")
    return None

def read_jsonl(data):
    return [json.loads(line) for line in data.splitlines() if line.strip()]

def input_rows(name):
    # Use the chosen bundle first, including its original nested-ZIP layout.
    for root in INPUT_ROOTS:
        direct = root / "text_inputs" / name
        if direct.is_file():
            return read_jsonl(direct.read_bytes())
        nested = root / TEXT_ZIP
        if nested.is_file():
            with ZipFile(nested) as archive:
                return read_jsonl(archive.read(name))
    path = find_input(name)
    if path is not None:
        return read_jsonl(path.read_bytes())
    with ZipFile(find_input(TEXT_ZIP, required=True)) as archive:
        return read_jsonl(archive.read(name))


def index_rows(rows, field="clip_id"):
    indexed = {row[field]: row for row in rows}
    if len(indexed) != len(rows):
        raise ValueError(f"Duplicate {field} values in an input.")
    return indexed

def text_probabilities(row):
    status = row.get("status", "ok")
    usable = row.get("text_usable", status in {"ok", "partial", "truncated"})
    if not usable or status not in {"ok", "partial", "truncated"}:
        return None
    order = row["score_labels"]
    scores = np.asarray(row["scores"], dtype=float)
    assert len(order) == len(LABELS) and set(order) == set(LABELS)
    assert scores.shape == (len(LABELS),) and np.isfinite(scores).all()
    scores = scores[[order.index(label) for label in LABELS]]
    if row.get("score_type", "probabilities") == "logits":
        scores = softmax(scores)
    else:
        assert row.get("score_type", "probabilities") == "probabilities"
        assert (scores >= 0).all() and np.isclose(scores.sum(), 1, atol=1e-5)
        scores /= scores.sum()
    return scores



def optional_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return False


In [ ]:
TRAIN_TEXT_EXPORT = "train2000_transcripts.jsonl"
JAIS_PROGRESS = "train2000_jais_progress.jsonl"


def save_jsonl(path, records):
    staged = path.with_suffix(".tmp")
    with staged.open("w", encoding="utf-8") as output:
        for row in records:
            output.write(json.dumps(row, ensure_ascii=False) + "\n")
    staged.replace(path)


from contextlib import contextmanager
from importlib.metadata import PackageNotFoundError, version
from src.experiments.jais_pilot import transcript_row
from src.models.jais_adapter import JaisAdapter, MODEL_ID, MODEL_REVISION, decode_audio

CASABLANCA_PROGRESS = "casablanca_full_jais_progress.jsonl"


def install_jais_dependencies():
    packages = {"transformers": "5.16.1", "accelerate": "1.15.0", "soxr": "1.1.0"}
    missing = []
    for package, expected in packages.items():
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        if installed != expected:
            missing.append(f"{package}=={expected}")
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def load_jais():
    install_jais_dependencies()
    from transformers import AutoModelForCausalLM, AutoProcessor

    options = {"revision": MODEL_REVISION, "trust_remote_code": True, "cache_dir": "/tmp/adi-jais"}
    processor = AutoProcessor.from_pretrained(MODEL_ID, **options)
    # The pinned adapter expects the model on a single device, including on a T4 ×2 session.
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **options, dtype=torch.float16, device_map={"": 0})
    return JaisAdapter(model, processor, max_new_tokens=128)


@contextmanager
def open_jais_shard(source, revision, shard, token):
    from huggingface_hub import HfFileSystem, hf_hub_download
    if source == "UBC-NLP/NADI_2026_ADI20_micro" and SPARSE_HF_READS:
        with HfFileSystem(token=token).open(f"datasets/{source}@{revision}/{shard}", "rb", block_size=8 * 1024**2) as handle:
            yield handle
        return
    with TemporaryDirectory(prefix="jais-shard-") as directory:
        yield hf_hub_download(repo_id=source, repo_type="dataset", revision=revision,
                              filename=shard, local_dir=directory, token=token)


def transcribe_rows(rows, transcripts, progress_path, *, casablanca=False, deadline=float("inf")):
    import gc
    import pyarrow.parquet as pq

    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU for missing Jais transcripts or attach the transcript progress file from a completed run.")
    source, rev = ("UBC-NLP/Casablanca", CAS_REV) if casablanca else ("UBC-NLP/NADI_2026_ADI20_micro", ADI_REV)
    buckets = defaultdict(dict)
    for row in rows:
        shard, index = row["source_shard"], int(row["source_row"])
        assert row["source_dataset"] == source and row["source_revision"] == rev
        assert ("/validation-" in shard if casablanca else shard.startswith("data/train-"))
        assert index not in buckets[shard]
        buckets[shard][index] = row
    started = time.monotonic()
    adapter = load_jais()
    was_deterministic = torch.are_deterministic_algorithms_enabled()
    torch.use_deterministic_algorithms(False)
    token = optional_hf_token()
    pending = []
    total_done = 0
    checkpoint = 0

    def store_record(row, record):
        nonlocal total_done, checkpoint
        key = row["clip_id"]
        assert record["clip_id"] == key
        if casablanca:
            record.update(audio_sha256=row["audio_sha256"], jais_revision=MODEL_REVISION)
        transcripts[key] = record
        total_done += 1
        if total_done == min(200, len(rows)):
            elapsed = time.monotonic() - started
            estimate = elapsed * (len(rows) - total_done) / total_done / 3600
            print(f"Jais wrote {total_done} rows. Estimated remaining time {estimate:.2f} h.", flush=True)
        if total_done - checkpoint >= 64:
            save_jsonl(progress_path, transcripts.values())
            checkpoint = total_done

    def predict_pending():
        manifest_rows, waveforms, rates = zip(*pending)
        keys = [row["clip_id"] for row in manifest_rows]
        try:
            raw = adapter.predict(keys, waveforms, rates)
        except torch.cuda.OutOfMemoryError:
            if len(keys) == 1:
                raise
            raw = None
        if raw is None:
            torch.cuda.empty_cache()
            raw = [adapter.predict([key], [wave], [rate])[0] for key, wave, rate in zip(keys, waveforms, rates, strict=True)]
        for row, item in zip(manifest_rows, raw, strict=True):
            record = transcript_row(item)
            status = item["prediction"]["status"]
            # The original Casablanca 800 treated unknown Jais dialect tags as inference failures.
            if casablanca and status == "inference_error":
                record.update(transcript=None, transcript_status="inference_error", usable_for_text_model=False)
            store_record(row, record)
        pending.clear()

    try:
        for i, (shard, wanted) in enumerate(sorted(buckets.items()), 1):
            if time.monotonic() >= deadline:
                return False
            print(f"Jais shard {i}/{len(buckets)} | {len(wanted)} clips | {shard}", flush=True)
            with open_jais_shard(source, rev, shard, token) as path, pq.ParquetFile(path) as parquet:
                offset = 0
                for group in range(parquet.num_row_groups):
                    end = offset + parquet.metadata.row_group(group).num_rows
                    selected = sorted(index for index in wanted if offset <= index < end)
                    if selected:
                        position = offset
                        for batch in parquet.iter_batches(columns=["audio"], row_groups=[group], batch_size=32):
                            batch_end = position + batch.num_rows
                            for index in selected:
                                if position <= index < batch_end:
                                    if time.monotonic() >= deadline:
                                        return False
                                    row = wanted.pop(index)
                                    payload = batch.column(0).field("bytes")[index - position].as_py()
                                    if not payload or hashlib.sha256(payload).hexdigest() != row["audio_sha256"]:
                                        raise RuntimeError(f"Audio SHA256 mismatch for {row['clip_id']}")
                                    try:
                                        waveform, rate = decode_audio(payload)
                                    except (ValueError, OSError, RuntimeError):
                                        store_record(row, unavailable_audio(row["clip_id"]))
                                        continue
                                    pending.append((row, waveform, rate))
                                    if len(pending) == 2:
                                        predict_pending()
                            position = batch_end
                            if position > selected[-1]:
                                break
                    offset = end
                    if not wanted:
                        break
            if wanted:
                raise RuntimeError(f"Missing {len(wanted)} requested parquet rows in {shard}")
            if pending:
                predict_pending()
            save_jsonl(progress_path, transcripts.values())
            checkpoint = total_done
            print(f"Jais progress {total_done}/{len(rows)} | {(time.monotonic() - started) / 60:.1f} min", flush=True)
        return True
    finally:
        save_jsonl(progress_path, transcripts.values())
        del adapter
        gc.collect()
        torch.cuda.empty_cache()
        torch.use_deterministic_algorithms(was_deterministic)


def unavailable_audio(key):
    return {"clip_id": key, "transcript": None, "transcript_status": "audio_error",
            "usable_for_text_model": False, "refused": False, "truncated": None}


def prepare_full_casablanca(rows, deadline):
    saved = index_rows(input_rows("casablanca_800_jais_transcripts.jsonl"))
    indexed = index_rows(rows)
    assert len(saved) == 800 and set(saved) <= set(indexed)
    assert Counter(indexed[key]["label"] for key in saved) == dict.fromkeys(CASABLANCA_LABELS, 100)
    broken = [row for row in rows if row.get("preparation_status", "ok") != "ok"]
    assert len(broken) == 1 and broken[0]["clip_id"] not in saved
    expected = {key: row for key, row in indexed.items() if key not in saved and row.get("preparation_status", "ok") == "ok"}
    progress_path = WORK / CASABLANCA_PROGRESS
    paths = sorted({p for root in INPUT_ROOTS for p in root.rglob(CASABLANCA_PROGRESS) if p.is_file()})
    if progress_path.is_file():
        paths.insert(0, progress_path)
    generated = {}
    for path in paths:
        for key, row in index_rows(read_jsonl(path.read_bytes())).items():
            if (key not in expected or row.get("audio_sha256") != expected[key]["audio_sha256"]
                    or row.get("jais_revision") != MODEL_REVISION):
                raise ValueError(f"Jais progress in {path.name} does not match the frozen Casablanca inputs.")
            generated.setdefault(key, row)
    missing = [row for key, row in expected.items() if key not in generated]
    print(f"Casablanca: {len(saved)} archived transcripts, {len(generated)} resumed, {len(missing)} need Jais, {len(broken)} unreadable.")
    if missing and not transcribe_rows(missing, generated, progress_path, casablanca=True, deadline=deadline):
        print(f"Time cap reached. Attach {CASABLANCA_PROGRESS} from Output to resume.")
        return None
    save_jsonl(progress_path, generated.values())
    combined = {**saved, **generated, broken[0]["clip_id"]: unavailable_audio(broken[0]["clip_id"])}
    assert set(combined) == set(indexed)
    return [combined[row["clip_id"]] for row in rows]


def training_export(train):
    original = index_rows(input_rows("jais_train500_transcripts.jsonl"))
    assert len(original) == 500 and set(original) <= set(train)
    if TEXT_TRAIN_SIZE == 500:
        return original, "Saved Jais 500-clip training export"
    full_path = find_input(TRAIN_TEXT_EXPORT)
    if full_path is not None:
        full = index_rows(read_jsonl(full_path.read_bytes()))
        if set(full) != set(train):
            raise ValueError(f"{TRAIN_TEXT_EXPORT} must cover the exact 2,000 frozen training IDs.")
        return full, "Attached or previously generated Jais 2,000-clip training export"

    progress = find_input(JAIS_PROGRESS)
    if progress is not None:
        saved = index_rows(read_jsonl(progress.read_bytes()))
        if not set(saved) <= set(train):
            raise ValueError("Cached Jais progress has IDs outside the frozen training manifest.")
        original.update((key, row) for key, row in saved.items() if key not in original)
    remaining = [row for key, row in train.items() if key not in original]
    if remaining:
        transcribe_rows(remaining, original, WORK / JAIS_PROGRESS)
    assert len(original) == len(train)
    save_jsonl(WORK / TRAIN_TEXT_EXPORT, (original[key] for key in train))
    return original, "Saved 500 Jais transcripts plus regenerated 1,500 Jais training transcripts"


In [ ]:
from src.models import transcript_tfidf as tfidf

assert TEXT_TRAIN_SIZE in {500, 2000}, "Select 500 or 2000 text training clips."
train = index_rows(read_manifest("train_scaling_2000"))
archived_availability = index_rows(input_rows("haidar_training_text_availability.jsonl"))
assert set(archived_availability) == set(train)
train_transcripts, train_source = training_export(train)
train_records = [train_transcripts[key] for key in train if key in train_transcripts]
train_targets = {key: row["label"] for key, row in train.items()}
assert len(train_records) == TEXT_TRAIN_SIZE
assert Counter(train_targets[row["clip_id"]] for row in train_records) == {
    label: TEXT_TRAIN_SIZE // len(LABELS) for label in LABELS
}
availability_drift = sum(
    (tfidf.available_text(row) is not None) != archived_availability[row["clip_id"]]["text_usable"]
    for row in train_records
)
if TEXT_TRAIN_SIZE == 500:
    assert availability_drift == 0, "Archived 500-clip text availability changed."
text_config = json.loads((REPO / "configs/transcript_tfidf.json").read_text())
text_model, training_diagnostics = tfidf.fit_model(train_records, train_targets, LABELS, text_config, train_source)
print(f"Text fit used {training_diagnostics['retained_training_clips']}/{TEXT_TRAIN_SIZE} training transcripts.")
print(f"Usable-text flags differing from Haidar's archive: {availability_drift}.")

casablanca_manifest = read_manifest("casablanca_validation")
assert len(casablanca_manifest) == 6819 and set(row["label"] for row in casablanca_manifest) == CASABLANCA_LABELS
assert FULL_EVAL_HOURS > 0
full_eval_deadline = time.monotonic() + FULL_EVAL_HOURS * 3600
casablanca_transcripts = prepare_full_casablanca(casablanca_manifest, full_eval_deadline)
ready = casablanca_transcripts is not None

if ready:
    splits = {
        "ADI20_dev": ("dev_pilot", input_rows("jais_dev200_transcripts.jsonl")),
        "Casablanca": ("casablanca_validation", casablanca_transcripts),
    }
    banks, eval_rows, text_source = {}, {}, {}
    for split, (manifest, transcripts) in splits.items():
        manifest_rows = read_manifest(manifest)
        indexed = index_rows(transcripts)
        eval_rows[split] = [row for row in manifest_rows if row["clip_id"] in indexed]
        assert len(eval_rows[split]) == len(transcripts)
        support = Counter(row["label"] for row in eval_rows[split])
        if split == "Casablanca":
            assert len(transcripts) == 6819 and set(support) == CASABLANCA_LABELS
        else:
            assert len(manifest_rows) == len(transcripts) == 200
        scores, diagnostics = tfidf.predict_records(text_model, transcripts, f"jais-tfidf-{TEXT_TRAIN_SIZE}")
        for row, detail in zip(scores, diagnostics, strict=True):
            row["text_usable"] = detail["prediction_source"] == "text"
        banks[split] = index_rows(scores)
        text_source[split] = (f"{train_source}; archived 800 plus full pinned Jais transcription"
                              if split == "Casablanca" else f"{train_source}; saved {split} Jais transcripts")
        print(f"{split}: {len(eval_rows[split])} clips | {sum(row['text_usable'] for row in scores)} with usable text")


## 3. Reuse AraBEST features and encode any missing clips


In [ ]:
feature_config = clf.feature_config(protocol["encoder"])
provenance = lm.expected_provenance(feature_config)
write_cache = ae.FeatureCache(WORK / "reusable_adi_cache", provenance)

# Use attached caches in place. New parts go only into the writable cache.
read_caches = []
for config_path in sorted(path for root in INPUT_ROOTS for path in root.rglob("config.json")):
    index_path = config_path.with_name("index.jsonl")
    if config_path.parent.name != write_cache.root.name or not index_path.is_file():
        continue
    mounted = ae.FeatureCache(WORK, provenance)
    mounted.root, mounted.index_path = config_path.parent, index_path
    read_caches.append(mounted)

def feature_locations(rows):
    indices = [(cache.root, cache.index()) for cache in [*read_caches, write_cache]]
    locations = {}
    part_exists = {}
    for row in rows:
        clip_id = row["clip_id"]
        key = ae.clip_cache_key(clip_id, row["audio_sha256"], provenance)
        for root, index in indices:
            saved = index.get(clip_id)
            if saved is None or saved["cache_key"] != key:
                continue
            part = root / saved["part"]
            if part not in part_exists:
                part_exists[part] = part.is_file()
            if part_exists[part]:
                locations[clip_id] = (part, saved["row"], key)
                break
    return locations

def missing_features(rows):
    locations = feature_locations(rows)
    return [row for row in rows if row["clip_id"] not in locations]

def feature_batches(rows, batch_size=64):
    locations = feature_locations(rows)
    groups = defaultdict(list)
    for i, row in enumerate(rows):
        clip_id = row["clip_id"]
        part, position, key = locations[clip_id]
        groups[part].append((i, position, clip_id, key))
    for part, entries in groups.items():
        with np.load(part, allow_pickle=False) as data:
            values, ids, keys = data["features"], data["clip_ids"], data["cache_keys"]
            for start in range(0, len(entries), batch_size):
                batch = entries[start:start + batch_size]
                for _, position, clip_id, key in batch:
                    assert ids[position] == clip_id and keys[position] == key, f"Invalid cache row for {clip_id}"
                features = values[[item[1] for item in batch]].astype(np.float32, copy=False)
                assert np.isfinite(features).all(), f"Nonfinite features in {part}"
                yield [item[0] for item in batch], features


In [ ]:
from importlib.metadata import PackageNotFoundError, version

encoder = None

def get_encoder():
    global encoder
    if encoder is None:
        try:
            installed = version("speechbrain")
        except PackageNotFoundError:
            installed = None
        if installed != "1.1.2":
            subprocess.run([
                sys.executable, "-m", "pip", "install", "-q", "--no-deps", "speechbrain==1.1.2",
            ], check=True)
        optional = {"hyperpyyaml": "hyperpyyaml", "sentencepiece": "sentencepiece", "joblib": "joblib"}
        needed = [package for module, package in optional.items() if importlib.util.find_spec(module) is None]
        if needed:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)
        encoder = ae.AcousticEncoder.from_pretrained(
            feature_config, device="cuda:0", cache_dir="/tmp/adi-hf-model"
        )
        clf.frozen_summary(encoder)
        assert json.loads(json.dumps(encoder.provenance())) == provenance, "Encoder provenance mismatch"
    return encoder

def encode_shard(handle, rows, scratch, label, deadline, on_encoded):
    import pyarrow.parquet as pq

    wanted = {int(row["source_row"]): row for row in rows}
    assert len(wanted) == len(rows), "Duplicate row in the pinned manifest"
    with pq.ParquetFile(handle) as parquet:
        offset = 0
        for group in range(parquet.num_row_groups):
            end = offset + parquet.metadata.row_group(group).num_rows
            selected = [i for i in wanted if offset <= i < end]
            if selected:
                position = offset
                for batch in parquet.iter_batches(
                    row_groups=[group], columns=["audio"], batch_size=EXTRACT_CHUNK
                ):
                    if time.monotonic() >= deadline:
                        return False
                    batch_end = position + batch.num_rows
                    block = []
                    for i in selected:
                        if position <= i < batch_end:
                            row = wanted.pop(i)
                            payload = batch.column(0).field("bytes")[i - position].as_py()
                            if not payload or hashlib.sha256(payload).hexdigest() != row["audio_sha256"]:
                                raise RuntimeError(f"Audio SHA256 mismatch for {row['clip_id']}")
                            path = ae.resolve_audio_path(scratch, row["audio_path"])
                            path.parent.mkdir(parents=True, exist_ok=True)
                            path.write_bytes(payload)
                            block.append(row)
                    if block:
                        _, failures = clf.prepare_features(
                            get_encoder(), write_cache, block, scratch,
                            {"batch_size": EXTRACT_BATCH, "chunk_size": EXTRACT_CHUNK}, label,
                        )
                        if failures:
                            with (WORK / "extraction_failures.jsonl").open("a", encoding="utf-8") as log:
                                for failure in failures:
                                    log.write(json.dumps(failure) + "\n")
                            raise RuntimeError(f"{len(failures)} failed features. See extraction_failures.jsonl.")
                        for row in block:
                            (scratch / row["audio_path"]).unlink()
                        on_encoded(len(block))
                    position = batch_end
                    if position > max(selected):
                        break
            offset = end
            if not wanted:
                break
    if wanted:
        raise RuntimeError(f"Pinned parquet did not contain {len(wanted)} requested rows.")
    return True

def fill_features(rows, label, deadline, limit=None):
    missing = sorted(
        missing_features(rows),
        key=lambda row: (row["source_dataset"], row["source_shard"], row["source_row"]),
    )
    if not missing:
        print(f"{label}: reusing all {len(rows)} cached clips")
        return True
    if not torch.cuda.is_available():
        raise RuntimeError("Select GPU T4 x2 or attach feature caches for the missing rows.")

    from huggingface_hub import HfFileSystem, hf_hub_download

    target = missing if limit is None else missing[:limit]
    buckets = defaultdict(list)
    for row in target:
        source, revision, shard = row["source_dataset"], row["source_revision"], row["source_shard"]
        if source == "UBC-NLP/Casablanca":
            assert revision == CAS_REV and "validation" in shard.lower() and "test" not in shard.lower()
        else:
            assert source == "UBC-NLP/NADI_2026_ADI20_micro" and revision == ADI_REV
        buckets[source, revision, shard].append(row)

    token = optional_hf_token()
    hub = HfFileSystem(token=token)
    started = time.monotonic()
    encoded = 0
    probe_size = min(PROBE_CLIPS, len(target))

    def record(count):
        nonlocal encoded
        previous = encoded
        encoded += count
        if previous < probe_size <= encoded:
            elapsed = time.monotonic() - started
            remaining_hours = elapsed * (len(missing) - encoded) / encoded / 3600
            print(
                f"{label}: {encoded} newly encoded in {elapsed / 60:.1f} min. "
                f"Rough estimate for remaining {len(missing) - encoded}: {remaining_hours:.2f} h.",
                flush=True,
            )

    for i, ((source, revision, shard), clips) in enumerate(buckets.items(), 1):
        if time.monotonic() >= deadline:
            return False
        print(f"{label}: shard {i}/{len(buckets)} | {len(clips)} clips | {shard}", flush=True)
        with TemporaryDirectory(prefix="adi-shard-") as directory:
            path = Path(directory)
            scratch = path / "audio"
            scratch.mkdir()
            if SPARSE_HF_READS and source == "UBC-NLP/NADI_2026_ADI20_micro":
                with hub.open(f"datasets/{source}@{revision}/{shard}", "rb", block_size=8 * 1024**2) as handle:
                    finished = encode_shard(handle, clips, scratch, label, deadline, record)
            else:
                local = hf_hub_download(
                    repo_id=source, repo_type="dataset", revision=revision, filename=shard,
                    local_dir=path / "parquet", token=token,
                )
                finished = encode_shard(local, clips, scratch, label, deadline, record)
            if not finished:
                return False
    assert not missing_features(target), f"{label}: feature cache is incomplete"
    return len(target) == len(missing)

if ready:
    wanted = [row for rows in eval_rows.values() for row in rows if row.get("preparation_status", "ok") == "ok"]
    missing = missing_features(wanted)
    print(f"Acoustic coverage: {len(wanted) - len(missing)}/{len(wanted)} cached.")
    if missing:
        if not EXTRACT_MISSING:
            raise RuntimeError("Attach the earlier acoustic caches or enable EXTRACT_MISSING with a GPU.")
        ready = fill_features(wanted, "Fusion evaluation audio", full_eval_deadline)
        if not ready:
            print("Time cap reached. Attach the saved acoustic cache from Output to resume.")
    encoder = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 4. Compare acoustic-only, text-only and fusion


In [ ]:
if ready:
    def load_arm(archive, arm, seeds):
        with np.load(io.BytesIO(archive.read(f"{arm}/zscore_stats.npz")), allow_pickle=False) as stats:
            mean, std = stats["mean"], stats["std"]
            assert stats["layer_names"].tolist() == provenance["layer_names"]
        assert mean.shape == std.shape == (26, 848)
        assert np.isfinite(mean).all() and np.isfinite(std).all() and (std > 0).all()
        models = {}
        for seed in seeds:
            data = archive.read(f"{arm}/head_{seed}.pt")
            checkpoint = torch.load(io.BytesIO(data), map_location="cpu", weights_only=True)
            assert checkpoint["labels"] == LABELS and checkpoint["layers"] == provenance["layer_names"]
            assert checkpoint["seed"] == seed and checkpoint["train_size"] == int(arm.split("_")[1])
            assert clf.feature_config(checkpoint["config"]["encoder"]) == feature_config
            model = lm.build_model((1, 26, 848), len(LABELS), 0)
            model.load_state_dict(checkpoint["state_dict"])
            models[seed] = model.eval()
        return mean, std, models


    def acoustic_scores(rows, arms):
        positions = [i for i, row in enumerate(rows) if row.get("preparation_status", "ok") == "ok"]
        readable = [rows[i] for i in positions]
        audio = {(arm, seed): np.zeros((len(rows), len(LABELS))) for arm, (_, _, models) in arms.items() for seed in models}
        with torch.inference_mode():
            for indices, raw in feature_batches(readable, batch_size=128):
                target = [positions[i] for i in indices]
                for arm, (mean, std, models) in arms.items():
                    x = torch.from_numpy((raw - mean) / std)
                    for seed, model in models.items():
                        logits = model(x).numpy()
                        if not np.isfinite(logits).all():
                            raise ValueError(f"Nonfinite acoustic logits for {arm}, seed {seed}.")
                        audio[arm, seed][target] = softmax(logits, axis=1)
        return audio


    def fuse_scores(audio, text, usable, weight):
        log_audio = np.log(audio + 1e-9)
        combined = log_audio.copy()
        combined[usable] += weight * np.log(text[usable] + 1e-9)
        fused = softmax(combined, axis=1)
        fused[~usable] = audio[~usable]
        return fused


    def predictions(scores):
        return np.where(scores.sum(axis=1) > 0, label_array[scores.argmax(axis=1)], "FAILURE")


    def metrics(gold, pred):
        present = sorted(set(gold))
        jor = gold == "JOR"
        jor_predicted = pred == "JOR"
        return {
            "accuracy": float((gold == pred).mean()),
            "macro_f1": f1_score(gold, pred, labels=present, average="macro", zero_division=0),
            "jor_n": int(jor.sum()),
            "jor_recall": float((pred[jor] == "JOR").mean()) if jor.any() else None,
            "jor_precision": float((gold[jor_predicted] == "JOR").mean()) if jor_predicted.any() else 0.0,
            "jor_to_syr": int((pred[jor] == "SYR").sum()),
            "n": len(gold),
            "coverage": float((pred != "FAILURE").mean()),
        }


    records, paired = [], []
    label_array = np.asarray(LABELS)
    with ZipFile(head_archive) as archive:
        arms = {arm: load_arm(archive, arm, protocol["seeds"]) for arm in ("train_4000", "train_8000")}
        for split, rows in eval_rows.items():
            gold = np.asarray([row["label"] for row in rows])
            ids = [row["clip_id"] for row in rows]
            text = np.zeros((len(ids), len(LABELS)))
            for i, key in enumerate(ids):
                p = text_probabilities(banks[split][key])
                if p is not None:
                    text[i] = p
            text_available = text.sum(axis=1) > 0
            text_pred = predictions(text)
            records.append(dict(split=split, arm="all", seed=None, variant="text_only", **metrics(gold, text_pred)))
            text_frame = pd.DataFrame({"clip_id": ids, "gold": gold, "prediction": text_pred, "text_usable": text_available})
            text_frame.join(pd.DataFrame(text, columns=LABELS)).to_csv(RUN / f"{split}_text_scores.csv", index=False)
            print(f"{split}: {text_available.sum()}/{len(ids)} clips have usable text")

            for (arm, seed), audio in acoustic_scores(rows, arms).items():
                audio_pred = predictions(audio)
                usable = text_available & (audio_pred != "FAILURE")
                fused = fuse_scores(audio, text, usable, FUSION_WEIGHT)
                fused_pred = predictions(fused)
                saved = pd.read_csv(io.BytesIO(archive.read(f"{arm}/{seed}/{split}_predictions.csv"))).set_index("clip_id")
                saved = saved.loc[ids]
                assert np.array_equal(saved["gold"].to_numpy(), gold)
                drift = int((saved["prediction"].to_numpy() != audio_pred).sum())
                if drift:
                    print(f"{split} {arm} {seed}: {drift} acoustic predictions differ from the archive; paired scores use recomputed audio.")
                stem = f"{split}_{arm}_{seed}"
                for variant, pred, scores in (("audio_only", audio_pred, audio), ("fusion", fused_pred, fused)):
                    records.append(dict(split=split, arm=arm, seed=seed, variant=variant, **metrics(gold, pred)))
                    frame = pd.DataFrame({"clip_id": ids, "gold": gold, "prediction": pred, "text_usable": text_available})
                    frame.join(pd.DataFrame(scores, columns=LABELS)).to_csv(RUN / f"{stem}_{variant}_scores.csv", index=False)
                    class_order = LABELS + ["FAILURE"]
                    counts = confusion_matrix(gold, pred, labels=class_order)
                    pd.DataFrame(counts, index=class_order, columns=class_order).to_csv(RUN / f"{stem}_{variant}_confusion.csv")
                a, f = audio_pred == gold, fused_pred == gold
                paired.append(dict(split=split, arm=arm, seed=seed, archive_audio_disagreements=drift,
                                   corrected=int((~a & f).sum()), broken=int((a & ~f).sum()),
                                   accuracy_delta_pp=100 * (f.mean() - a.mean()),
                                   macro_f1_delta_pp=100 * (metrics(gold, fused_pred)["macro_f1"] - metrics(gold, audio_pred)["macro_f1"]),
                                   jor_to_syr_audio=int(((gold == "JOR") & (audio_pred == "SYR")).sum()),
                                   jor_to_syr_fused=int(((gold == "JOR") & (fused_pred == "SYR")).sum())))

    results = pd.DataFrame(records)
    results.to_csv(RUN / "results.csv", index=False)
    deltas = pd.DataFrame(paired)
    deltas.to_csv(RUN / "paired_deltas.csv", index=False)
    display((100 * results.groupby(["split", "arm", "variant"])[
        ["accuracy", "macro_f1", "jor_recall", "jor_precision", "coverage"]
    ].mean()).round(2))
    display(deltas)
    (RUN / "fusion_protocol.json").write_text(json.dumps({
        "fixed_text_weight": FUSION_WEIGHT, "weight_source": "Haidar EEC-27 training OOF preset",
        "text_inputs": text_source,
        "text_training_n": TEXT_TRAIN_SIZE,
        "text_training_id_sha256": hashlib.sha256("\n".join(row["clip_id"] for row in train_records).encode()).hexdigest(),
        "text_training_diagnostics": training_diagnostics,
        "haidar_text_availability_disagreements": availability_drift,
        "casablanca_eval_source": "Full frozen Casablanca validation. 800 archived transcript records, 6,018 newly covered readable clips, 1 unreadable row.",
        "rows": {split: len(rows) for split, rows in eval_rows.items()},
        "audio_checkpoint_protocol": protocol,
        "casablanca_acoustic_archive_agreement": int(deltas.loc[deltas["split"] == "Casablanca", "archive_audio_disagreements"].sum()) == 0,
        "fallback": "Use exact acoustic scores when text is unusable. Acoustic failures remain failures.",
        "casablanca_reference_set_n": 6819,
        "jais_model_id": MODEL_ID,
        "jais_model_revision": MODEL_REVISION,
        "jais_max_new_tokens": 128,
    }, indent=2))
    print("Download:", shutil.make_archive(str(WORK / "adi20_casablanca_full_fusion_results"), "zip", RUN))
else:
    print("Evaluation paused before all 6,819 Casablanca rows were ready. Save the progress and caches from Output.")
